# MOD-12930 — balanced full suite

Same contenders and axes as the main (imbalanced) suite, but the text query of every
(size, depth) cell is **calibrated** so the SEARCH mirror's p50 matches the VSIM mirror's
(±20%, geometric bisection on the target match count; the achieved ratio is recorded).

**Why balance:** with skewed branches, `hybrid ≈ max + C` and `hybrid ≈ sum + C'` fit the
same measurement (sum − max = the small branch ≈ noise), so neither concurrency nor
overhead is identifiable. Equal branches maximize sum − max, so the workers-0/workers-N
pair separates "branches overlap" from "serial overhead C" and yields C twice, independently.

**Matrix:** size (10K/100K/500K) × depth K/W (10/20, 50/100, 250/500 — giving the
C(WINDOW) curve) × workers=2 × fields (none/title+text) × 4 contenders.
A cell whose calibration cannot reach balance is recorded with `balanced=False`.

In [1]:
import json, os
import pandas as pd

# Set REUSE_RESULTS=1 to re-render the tables from a previously saved run
# without re-executing the benchmark.
if os.environ.get('REUSE_RESULTS') and os.path.exists('results_balanced_full.json'):
    d = json.load(open('results_balanced_full.json'))
    results, gates, meta = d['results'], d['gates'], d['meta']
    print('reusing saved results_balanced_full.json')
else:
    import bench_lib as B
    import balanced_lib as BAL
    titles, texts, emb, corpus_max = B.load_data()
    results, gates, meta = BAL.run_balanced_full(titles, texts, emb, corpus_max)
    with open('results_balanced_full.json', 'w') as f:
        json.dump(dict(meta=meta, results=results, gates=gates), f, indent=2, default=str)
    pd.DataFrame(results).to_csv('results_balanced_full.csv', index=False)
    print('saved results_balanced_full.json / .csv')

reusing saved results_balanced_full.json


## Calibration & gates

`balance_ratio` = search p50 / vsim p50 at calibration (1.0 = perfect balance; a cell counts as balanced within ±20%).

In [2]:
pd.DataFrame(gates)

,size,k,window,matches_mean,balance_ratio,gate_linear,gate_rrf
0,10000,10,20,1192.50000,1.08,16/16,16/16
1,10000,50,100,644.90625,0.88,16/16,16/16
2,10000,250,500,2970.34375,0.98,16/16,16/16
3,100000,10,20,1689.31250,1.07,16/16,16/16
4,100000,50,100,5626.56250,1.03,16/16,16/16
5,100000,250,500,12625.46875,0.92,16/16,16/16
6,500000,10,20,1436.87500,0.93,16/16,16/16
7,500000,50,100,2781.40625,1.02,16/16,16/16
8,500000,250,500,9446.31250,1.02,16/16,16/16


## p50 latency (ms)

In [3]:
df = pd.DataFrame(results)
pivot = df.pivot_table(index=['size', 'window', 'workers', 'fields'], columns='contender',
                       values='p50_ms', sort=False).round(2)
pivot[['hybrid_linear', 'hybrid_rrf', 'search_branch', 'vsim_branch']]

contender                         hybrid_linear  hybrid_rrf  search_branch  \
size   window workers fields                                                 
10000  20     2       none                 0.37        0.34           0.20   
                      title+text           0.41        0.40           0.26   
       100    2       none                 0.54        0.56           0.24   
                      title+text           0.63        0.66           0.30   
       500    2       none                 2.36        2.31           1.04   
                      title+text           2.77        2.66           1.01   
100000 20     2       none                 0.42        0.43           0.25   
                      title+text           0.43        0.46           0.35   
       100    2       none                 0.89        0.90           0.40   
                      title+text           1.05        1.03           0.53   
       500    2       none                 3.16        3.15           1.37   
                      title+text           3.54        3.68           1.41   
500000 20     2       none                 0.41        0.43           0.30   
                      title+text           0.45        0.50           0.41   
       100    2       none                 0.94        0.93           0.54   
                      title+text           1.04        1.06           0.60   
       500    2       none                 3.53        3.49           1.69   
                      title+text           3.91        3.95           1.76   

contender                         vsim_branch  
size   window workers fields                   
10000  20     2       none               0.16  
                      title+text         0.22  
       100    2       none               0.24  
                      title+text         0.36  
       500    2       none               0.97  
                      title+text         0.90  
100000 20     2       none               0.23  
                      title+text         0.27  
       100    2       none               0.43  
                      title+text         0.51  
       500    2       none               1.46  
                      title+text         1.52  
500000 20     2       none               0.23  
                      title+text         0.32  
       100    2       none               0.49  
                      title+text         0.57  
       500    2       none               1.66  
                      title+text         1.71

## C — the serial overhead, and the C(WINDOW) curve

fields=none. `C_w0 = hybrid − (search+vsim)`, `C_w6 = hybrid − max(search,vsim)`
(mean latency). If depletion truly overlaps and C is genuinely serial, the two agree.
`C/max` says how the overhead compares to running one entire branch.

In [4]:
m = df[df.fields == 'none'].pivot_table(index=['size', 'window', 'workers'],
        columns='contender', values='mean_ms', sort=False)
mx = m[['search_branch', 'vsim_branch']].max(axis=1)
sm = m[['search_branch', 'vsim_branch']].sum(axis=1)
c = pd.DataFrame({'hybrid': m['hybrid_linear'], 'max_branch': mx, 'sum_branch': sm})
c['C_ms'] = (c['hybrid'] - c['sum_branch']).where(
    c.index.get_level_values('workers') == 0, c['hybrid'] - c['max_branch'])
c['C_pct_of_max'] = (100 * c['C_ms'] / c['max_branch'])
c['gain_hint'] = None
c.round(2)

hybrid  max_branch  sum_branch  C_ms  C_pct_of_max  \
size   window workers                                                       
10000  20     2          0.39        0.23        0.40  0.16         71.72   
       100    2          0.55        0.25        0.49  0.30        120.43   
       500    2          2.40        1.04        2.01  1.36        130.78   
100000 20     2          0.45        0.27        0.50  0.18         64.54   
       100    2          0.97        0.51        0.94  0.46         88.86   
       500    2          3.31        1.52        2.97  1.79        117.64   
500000 20     2          0.43        0.31        0.54  0.12         38.23   
       100    2          0.98        0.59        1.08  0.39         66.29   
       500    2          3.65        1.75        3.38  1.90        108.50   

                      gain_hint  
size   window workers            
10000  20     2            None  
       100    2            None  
       500    2            None  
100000 20     2            None  
       100    2            None  
       500    2            None  
500000 20     2            None  
       100    2            None  
       500    2            None

### Parallelism gain per depth (`p50(w0)/p50(w_max)`, fields=none) — only when both were measured

In [5]:
ws = sorted(df['workers'].unique())
if len(ws) > 1 and 0 in ws:
    p = df[df.fields == 'none'].pivot_table(index=['size', 'window', 'contender'],
            columns='workers', values='p50_ms', sort=False)
    p.columns = [f'w{c}' for c in p.columns]
    p['gain'] = (p['w0'] / p[f'w{max(ws)}']).round(2)
    display(p.round(2))
else:
    print(f'single workers setting measured ({ws}) — gain not applicable')

single workers setting measured ([np.int64(2)]) — gain not applicable


## Merger scenario sweep

C = hybrid − max(search, vsim) with window, size and selectivity varied **independently**
(the calibrated matrix above ties |matches| to the vector branch's latency). Grid:
size × window × match-fraction (1% / 10% / 50% of corpus), fields=none.

In [6]:
if os.environ.get('REUSE_RESULTS') and os.path.exists('results_merger_sweep.json'):
    sw = json.load(open('results_merger_sweep.json'))
    sweep_results, sweep_meta = sw['results'], sw['meta']
    print('reusing saved results_merger_sweep.json')
else:
    import bench_lib as B
    from merger_sweep import run_sweep
    sweep_results, sweep_meta = run_sweep(*B.load_data())

sdf = pd.DataFrame(sweep_results)
sdf.pivot_table(index='window', columns=['size', 'match_frac'], values='C_ms').round(2)

reusing saved results_merger_sweep.json


size       10000              100000             500000            
match_frac   0.01  0.10  0.50   0.01  0.10  0.50   0.01  0.10  0.50
window                                                             
20           0.04  0.16  0.07   0.14  0.37  0.55   0.49  1.46  5.91
100          0.29  0.31  0.34   0.41  0.53  1.06   0.55  2.06  0.68
500          0.68  0.92  1.32   0.91  1.79  2.25   1.80  6.28  5.55